# ACRIN NSCLC — Leakage-Corrected Canonical Pipeline
Fixes applied following external review of the prior version:

1. **ComBat leakage (critical, fixed):** the prior version fit ComBat on the FULL cohort — including
   `label_2yr` as a protected covariate — before any train/test split. This is documented leakage
   (Da-ano et al. 2020; Orlhac et al. 2022). Fixed by switching to `neuroHarmonize`
   (`harmonizationLearn`/`harmonizationApply`), which supports fitting the batch-effect model on
   training data ONLY, then applying those fixed parameters to the held-out test fold using that
   fold's own batch/outcome values -- no information from other patients crosses the train/test
   boundary. Harmonization is now a pipeline step, refit inside every CV fold.
2. **SMOTE reordered to run AFTER feature selection**, not before -- so RFE selects features from
   real patient data, not synthetic interpolated samples.
3. **RFE's base estimator now matches the classifier being evaluated** (RandomForest, GradientBoosting,
   XGBoost, Logistic all support this via `feature_importances_`/`coef_`); SVM is the one necessary
   exception (default RBF kernel has neither), so Logistic is used as RFE's estimator for that case only.
4. **RepeatedStratifiedKFold (5 splits x 10 repeats = 50 fold evaluations)** instead of a single 5-fold
   pass, for a materially more stable variance estimate at N=48.
5. **SMOTE k_neighbors guard**: dynamically capped below the training fold's minority class count,
   to avoid silent failure/misbehavior when a fold's minority class is very small.
6. **Permutation test** added for the top candidate result, to check whether it is distinguishable
   from chance given 30 total dataset/classifier comparisons.

**Remaining, disclosed (not fixed) limitations**, stated explicitly rather than hidden:
- RFE(3) is an imposed constraint (Ma'am's instruction), not a data-driven/tuned choice.
- Correlation-filter threshold (0.90) is an assumption, not confirmed with Ma'am.
- Median imputation for the handful of missing raw values (37/27/5 out of ~10,752 per table) is
  still done globally before the CV loop, not per-fold -- a much lower-risk simplification than the
  ComBat leakage was, since it affects <0.5% of values, but disclosed here for completeness.
- No SHAP / calibration curves / learning curves are included.
- Only one feature-selection method (correlation filter -> RFE) and one classifier-tuning level
  (default hyperparameters) are tested.


In [ ]:
import numpy as np
import pandas as pd
import os
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt

from neuroHarmonize import harmonizationLearn, harmonizationApply
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.feature_selection import RFE
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, brier_score_loss
)

RANDOM_STATE = 42
N_SPLITS = 5
N_REPEATS = 10
N_FEATURES_RFE = 3          # imposed constraint (Ma'am's instruction), not data-driven
CORR_THRESHOLD = 0.90       # assumption for the feature-filtering step -- confirm with Ma'am
RFE_STEP = 10               # DISCLOSED SPEED/PRECISION TRADE-OFF: removes 10 features per RFE
                             # iteration instead of the default 1. Much faster (step=1 meant ~447
                             # refits of a 200-tree classifier PER FOLD for Model 2/3's ~450
                             # features), but coarser -- it does not backtrack, so a feature useful
                             # only in combination with others can be eliminated in an early batch
                             # cut that a step=1 pass would have kept. Given this pipeline's whole
                             # purpose is to defend a "top 3 features" claim, this trade-off is a
                             # real limitation, not a neutral implementation detail.
RAW_CSV = "radiomic_acrin_nsclc_dataset.csv"


## Step 1 — Load data, derive Modality / Timepoint / ROI_Type (unchanged from prior version)

In [ ]:
df = pd.read_csv(RAW_CSV)

def clean_column_names(df):
    df = df.copy()
    df.columns = (
        df.columns.str.replace(" ", "_").str.replace("-", "_")
        .str.replace("(", "", regex=False).str.replace(")", "", regex=False)
        .str.replace("[", "", regex=False).str.replace("]", "", regex=False)
        .str.replace("%", "pct", regex=False).str.replace("/", "_", regex=False)
    )
    return df

df = clean_column_names(df)
df["INFO_NameOfRoi"] = df["INFO_NameOfRoi"].astype(str).str.replace(" ", "_").str.replace("-", "_")
df["Modality"] = df["INFO_Series"].apply(lambda x: "PET" if str(x).strip().endswith("_PT") else ("CT" if str(x).strip().endswith("_CT") else "UNKNOWN"))
df["Timepoint"] = df["INFO_NameOfRoi"].apply(lambda x: "T1" if x.startswith("R1_") else ("T2" if x.startswith("R2_") else "UNKNOWN"))

def get_roi_type(roi_name):
    if "Dilate_less_" in roi_name: return "Peritumoral"
    elif "_Dilate" in roi_name: return "Combined"
    else: return "Tumoral"

df["ROI_Type"] = df["INFO_NameOfRoi"].apply(get_roi_type)
assert (df["Modality"] == "UNKNOWN").sum() == 0
assert (df["Timepoint"] == "UNKNOWN").sum() == 0
print("Loaded:", df.shape)


## Step 2 — Feature whitelist (broken/empty features excluded, confirmed via prior testing)

In [ ]:
keep_these = [
    "MORPHOLOGICAL_Volume(IBSI:RNU0)[cm3]", "MORPHOLOGICAL_ApproximateVolume(IBSI:YEKZ)[cm3]",
    "MORPHOLOGICAL_SurfaceArea(IBSI:C0JK)[cm2]", "MORPHOLOGICAL_SurfaceToVolumeRatio(IBSI:2PR5)[cm]",
    "MORPHOLOGICAL_Compactness1(IBSI:SKGS)[]", "MORPHOLOGICAL_Compactness2(IBSI:BQWJ)[]",
    "MORPHOLOGICAL_SphericalDisproportion(IBSI:KRCK)[]", "MORPHOLOGICAL_Sphericity(IBSI:QCFX)[]",
    "MORPHOLOGICAL_Asphericity(IBSI:25C7)[]", "MORPHOLOGICAL_CentreOfMassShift(IBSI:KLMA)[mm]",
    "MORPHOLOGICAL_Maximum3DDiameter(IBSI:L0JK)[mm]", "MORPHOLOGICAL_IntegratedIntensity(IBSI:99N0)[Intensity]",
    "INTENSITY-BASED_MeanIntensity(IBSI:Q4LE)[SUVbw]", "INTENSITY-BASED_IntensityVariance(IBSI:ECT3)[SUVbw]",
    "INTENSITY-BASED_IntensitySkewness(IBSI:KE2A)[SUVbw]", "INTENSITY-BASED_IntensityKurtosis(IBSI:IPH6)[SUVbw]",
    "INTENSITY-BASED_MedianIntensity(IBSI:Y12H)[SUVbw]", "INTENSITY-BASED_MinimumIntensity(IBSI:1GSF)[SUVbw]",
    "INTENSITY-BASED_10thIntensityPercentile(IBSI:QG58)[SUVbw]", "INTENSITY-BASED_50thIntensityPercentile(IBSI:Y12H)[SUVbw]",
    "INTENSITY-BASED_90thIntensityPercentile(IBSI:8DWT)[SUVbw]", "INTENSITY-BASED_MaximumIntensity(IBSI:84IY)[SUVbw]",
    "INTENSITY-BASED_IntensityInterquartileRange(IBSI:SALO)[SUVbw]", "INTENSITY-BASED_IntensityRange(IBSI:2OJQ)[SUVbw]",
    "INTENSITY-BASED_IntensityBasedMeanAbsoluteDeviation(IBSI:4FUA)[SUVbw]",
    "INTENSITY-BASED_IntensityBasedRobustMeanAbsoluteDeviation(IBSI:1128)[SUVbw]",
    "INTENSITY-BASED_IntensityBasedMedianAbsoluteDeviation(IBSI:N72L)[SUVbw]",
    "INTENSITY-BASED_IntensityBasedCoefficientOfVariation(IBSI:7TET)[SUVbw]",
    "INTENSITY-BASED_IntensityBasedQuartileCoefficientOfDispersion(IBSI:9S40)[SUVbw]",
    "INTENSITY-BASED_IntensityBasedEnergy(IBSI:N8CA)[SUVbw]", "INTENSITY-BASED_RootMeanSquareIntensity(IBSI:5ZWQ)[SUVbw]",
    "INTENSITY-HISTOGRAM_IntensityHistogramMean(IBSI:X6K6)[Intensity]", "INTENSITY-HISTOGRAM_IntensityHistogramVariance(IBSI:CH89)[Intensity]",
    "INTENSITY-HISTOGRAM_IntensityHistogramSkewness(IBSI:88K1)[Intensity]", "INTENSITY-HISTOGRAM_IntensityHistogramKurtosis(IBSI:C3I7)[Intensity]",
    "INTENSITY-HISTOGRAM_IntensityHistogramMedian(IBSI:WIFQ)[Intensity]", "INTENSITY-HISTOGRAM_IntensityHistogramMinimumGreyLevel(IBSI:1PR8)[Intensity]",
    "INTENSITY-HISTOGRAM_IntensityHistogram10thPercentile(IBSI:GPMT)[]", "INTENSITY-HISTOGRAM_IntensityHistogram90thPercentile(IBSI:OZ0C)[]",
    "INTENSITY-HISTOGRAM_IntensityHistogramMaximumGreyLevel(IBSI:3NCY)[Intensity]", "INTENSITY-HISTOGRAM_IntensityHistogramMode(IBSI:AMMC)[Intensity]",
    "INTENSITY-HISTOGRAM_IntensityHistogramInterquartileRange(IBSI:WR0O)[Intensity]", "INTENSITY-HISTOGRAM_IntensityHistogramRange(IBSI:5Z3W)[Intensity]",
    "INTENSITY-HISTOGRAM_IntensityHistogramMeanAbsoluteDeviation(IBSI:D2ZX)[Intensity]",
    "INTENSITY-HISTOGRAM_IntensityHistogramRobustMeanAbsoluteDeviation(IBSI:WRZB)[Intensity]",
    "INTENSITY-HISTOGRAM_IntensityHistogramMedianAbsoluteDeviation(IBSI:4RNL)[Intensity]",
    "INTENSITY-HISTOGRAM_IntensityHistogramCoefficientOfVariation(IBSI:CWYJ)[Intensity]",
    "INTENSITY-HISTOGRAM_IntensityHistogramQuartileCoefficientOfDispersion(IBSI:SLWD)[Intensity]",
    "INTENSITY-HISTOGRAM_IntensityHistogramEntropyLog2(IBSI:TLU2)[Intensity]", "INTENSITY-HISTOGRAM_Uniformity(IBSI:BJ5W)[Intensity]",
    "INTENSITY-HISTOGRAM_MaximumHistogramGradient(IBSI:12CE)[Intensity]", "INTENSITY-HISTOGRAM_MaximumHistogramGradientGreyLevel(IBSI:8E6O)[Intensity]",
    "INTENSITY-HISTOGRAM_MinimumHistogramGradient(IBSI:VQB3)[Intensity]", "INTENSITY-HISTOGRAM_MinimumHistogramGradientGreyLevel(IBSI:RHQZ)[Intensity]",
    "GLCM_JointMaximum(IBSI:GYBY)", "GLCM_JointAverage(IBSI:60VM)", "GLCM_JointVariance(IBSI:UR99)",
    "GLCM_JointEntropyLog2(IBSI:TU9B)", "GLCM_DifferenceAverage(IBSI:TF7R)", "GLCM_DifferenceVariance(IBSI:D3YU)",
    "GLCM_DifferenceEntropy(IBSI:NTRS)", "GLCM_SumAverage(IBSI:ZGXS)", "GLCM_SumVariance(IBSI:OEEB)",
    "GLCM_SumEntropy(IBSI:P6QZ)", "GLCM_AngularSecondMoment(IBSI:8ZQL)", "GLCM_Contrast(IBSI:ACUI)",
    "GLCM_Dissimilarity(IBSI:8S9J)", "GLCM_InverseDifference(IBSI:IB1Z)", "GLCM_NormalisedInverseDifference(IBSI:NDRX)",
    "GLCM_InverseDifferenceMoment(IBSI:WF0Z)", "GLCM_NormalisedInverseDifferenceMoment(IBSI:1QCO)",
    "GLCM_InverseVariance(IBSI:E8JP)", "GLCM_Correlation(IBSI:NI2N)", "GLCM_Autocorrelation(IBSI:QWB0)",
    "GLCM_ClusterTendency(IBSI:DG8W)", "GLCM_ClusterShade(IBSI:7NFM)", "GLCM_ClusterProminence(IBSI:AE86)",
    "GLRLM_ShortRunsEmphasis(IBSI:22OV)", "GLRLM_LongRunsEmphasis(IBSI:W4KF)", "GLRLM_LowGreyLevelRunEmphasis(IBSI:V3SW)",
    "GLRLM_HighGreyLevelRunEmphasis(IBSI:G3QZ)", "GLRLM_ShortRunLowGreyLevelEmphasis(IBSI:HTZT)",
    "GLRLM_ShortRunHighGreyLevelEmphasis(IBSI:GD3A)", "GLRLM_LongRunLowGreyLevelEmphasis(IBSI:IVPO)",
    "GLRLM_LongRunHighGreyLevelEmphasis(IBSI:3KUM)", "GLRLM_GreyLevelNonUniformity(IBSI:R5YN)",
    "GLRLM_RunLengthNonUniformity(IBSI:W92Y)", "GLRLM_RunPercentage(IBSI:9ZK5)", "GLRLM_GreyLevelVariance(IBSI:8CE5)",
    "GLRLM_RunLengthVariance(IBSI:SXLW)", "GLRLM_RunEntropy(IBSI:HJ90)",
    "NGTDM_Coarseness(IBSI:QCDE)", "NGTDM_Contrast(IBSI:65HE)", "NGTDM_Busyness(IBSI:NQ30)",
    "NGTDM_Complexity(IBSI:HDEZ)", "NGTDM_Strength(IBSI:1X9X)",
    "GLSZM_SmallZoneEmphasis(IBSI:5QRC)", "GLSZM_LargeZoneEmphasis(IBSI:48P8)", "GLSZM_LowGrayLevelZoneEmphasis(IBSI:XMSY)",
    "GLSZM_HighGrayLevelZoneEmphasis(IBSI:5GN9)", "GLSZM_SmallZoneLowGreyLevelEmphasis(IBSI:5RAI)",
    "GLSZM_SmallZoneHighGreyLevelEmphasis(IBSI:HW1V)", "GLSZM_LargeZoneLowGreyLevelEmphasis(IBSI:YH51)",
    "GLSZM_LargeZoneHighGreyLevelEmphasis(IBSI:J17V)", "GLSZM_GreyLevelNonUniformity(IBSI:JNSA)",
    "GLSZM_NormalisedGreyLevelNonUniformity(IBSI:Y1RO)", "GLSZM_ZoneSizeNonUniformity(IBSI:4JP3)",
    "GLSZM_NormalisedZoneSizeNonUniformity(IBSI:VB3A)", "GLSZM_ZonePercentage(IBSI:P30P)",
    "GLSZM_GreyLevelVariance(IBSI:BYLV)", "GLSZM_ZoneSizeVariance(IBSI:3NSA)", "GLSZM_ZoneSizeEntropy(IBSI:GU8N)",
]

def clean_list(col_list):
    cleaned = []
    for col in col_list:
        col = (col.replace(" ", "_").replace("-", "_").replace("(", "").replace(")", "")
                  .replace("[", "").replace("]", "").replace("%", "pct").replace("/", "_"))
        cleaned.append(col)
    return cleaned

keep_these_clean = clean_list(keep_these)
feature_cols_candidate = [c for c in keep_these_clean if c in df.columns]
print(f"{len(feature_cols_candidate)} whitelisted features found.")


## Step 3 — Batch label (GE vs NonGE, verified consistent across PET+CT)

In [ ]:
pet_mfg = df[df["Modality"]=="PET"].drop_duplicates("INFO_PatientID")[["INFO_PatientID","Scanner_Manufacturer"]]
ct_mfg = df[df["Modality"]=="CT"].drop_duplicates("INFO_PatientID")[["INFO_PatientID","Scanner_Manufacturer"]]
mfg_check = pet_mfg.merge(ct_mfg, on="INFO_PatientID", suffixes=("_PET","_CT"))
assert (mfg_check["Scanner_Manufacturer_PET"]=="GE MEDICAL SYSTEMS").equals(mfg_check["Scanner_Manufacturer_CT"]=="GE MEDICAL SYSTEMS")

batch_map = pet_mfg.set_index("INFO_PatientID")["Scanner_Manufacturer"].apply(
    lambda x: "GE" if x == "GE MEDICAL SYSTEMS" else "NonGE"
).to_dict()
print(pd.Series(batch_map).value_counts())


## Step 3b — Batch/outcome confounding check
`label_2yr` is used as a protected covariate in ComBat (Step 6), which is only a sound modeling
choice if scanner batch and outcome are roughly independent. If they were strongly correlated
(e.g., one scanner used mostly at a site with systematically different patient outcomes), ComBat
would under-remove batch effects that happen to align with the outcome label, and harmonized-vs-
non-harmonized differences could partly reflect scanner artifact rather than biology. This check
must be run and reviewed before trusting any Harmonized result below.


In [ ]:
label_by_patient = df.drop_duplicates("INFO_PatientID").set_index("INFO_PatientID")["label_2yr"]
batch_series = pd.Series(batch_map)
crosstab = pd.crosstab(batch_series, label_by_patient.reindex(batch_series.index), margins=True)
print("Batch x Outcome crosstab:")
print(crosstab)
print()
survival_by_batch = label_by_patient.reindex(batch_series.index).groupby(batch_series).mean()
print("2-year survival rate by batch:")
print(survival_by_batch)
gap = abs(survival_by_batch.iloc[0] - survival_by_batch.iloc[1])
print(f"\nAbsolute survival-rate gap between batches: {gap:.3f}")
if gap > 0.20:
    print("WARNING: large batch/outcome gap -- ComBat's outcome-protective covariate may under-remove")
    print("batch effects confounded with the label. Harmonized results should be interpreted cautiously.")
else:
    print("Gap is modest -- reduces, but does not eliminate, the risk of outcome-confounded harmonization.")


## Step 4 — Build combined PET+CT tables (raw, NOT harmonized here -- harmonization now happens
INSIDE the CV loop, per fold, to fix the leakage issue).

In [ ]:
def build_combined_table(timepoint, roi_type):
    pet = df[(df["Modality"]=="PET") & (df["Timepoint"]==timepoint) & (df["ROI_Type"]==roi_type)]
    ct = df[(df["Modality"]=="CT") & (df["Timepoint"]==timepoint) & (df["ROI_Type"]==roi_type)]
    pet = pet[["INFO_PatientID", "label_2yr"] + feature_cols_candidate].drop_duplicates("INFO_PatientID")
    ct = ct[["INFO_PatientID"] + feature_cols_candidate].drop_duplicates("INFO_PatientID")
    pet = pet.rename(columns={c: f"PET_{c}" for c in feature_cols_candidate})
    ct = ct.rename(columns={c: f"CT_{c}" for c in feature_cols_candidate})
    merged = pet.merge(ct, on="INFO_PatientID", how="inner")
    merged["CombinedBatch"] = merged["INFO_PatientID"].map(batch_map)
    return merged

roi_types_to_build = ["Tumoral", "Peritumoral", "Combined"]
combined_tables = {}
for t in ["T1", "T2"]:
    for r in roi_types_to_build:
        combined_tables[(t, r)] = build_combined_table(t, r)

combined_feature_cols_candidate = [f"PET_{c}" for c in feature_cols_candidate] + [f"CT_{c}" for c in feature_cols_candidate]

def find_bad_cols(table, cols):
    bad = set()
    for c in cols:
        if table[c].notna().sum() == 0:
            bad.add(c); continue
        for b in table["CombinedBatch"].unique():
            if table.loc[table["CombinedBatch"]==b, c].nunique(dropna=True) <= 1:
                bad.add(c); break
    return bad

all_bad = set()
for table in combined_tables.values():
    all_bad |= find_bad_cols(table, combined_feature_cols_candidate)
combined_feature_cols = [c for c in combined_feature_cols_candidate if c not in all_bad]
print(f"Dropped {len(all_bad)} broken feature(s); proceeding with {len(combined_feature_cols)}.")

# NOTE: no global pre-CV imputation here anymore. Imputation is now handled entirely inside the
# pipeline -- by FoldSafeHarmonizer for harmonized runs (fit per-fold, feeds clean data to ComBat),
# and by the standalone SimpleImputer step for non-harmonized runs. This removes the small,
# previously-disclosed global-imputation leak entirely rather than leaving it as an exception.


## Step 5 — Compute delta (T2 - T1), build Model 1 / 2 / 3, KEEPING the batch label attached

In [ ]:
def compute_delta(t1, t2, roi_type, cols):
    merged = t1[["INFO_PatientID", "label_2yr", "CombinedBatch"] + cols].merge(
        t2[["INFO_PatientID"] + cols], on="INFO_PatientID", suffixes=("_T1","_T2"), how="inner"
    )
    delta = pd.DataFrame()
    delta["INFO_PatientID"] = merged["INFO_PatientID"]
    delta["label_2yr"] = merged["label_2yr"]
    delta["CombinedBatch"] = merged["CombinedBatch"]
    for c in cols:
        delta[f"{roi_type}_{c}_delta"] = merged[f"{c}_T2"] - merged[f"{c}_T1"]
    return delta

delta_by_roi = {r: compute_delta(combined_tables[("T1", r)], combined_tables[("T2", r)], r, combined_feature_cols) for r in roi_types_to_build}

def build_models(delta_dict):
    m1 = delta_dict["Tumoral"].copy()
    m2 = m1.merge(delta_dict["Peritumoral"].drop(columns=["label_2yr", "CombinedBatch"]), on="INFO_PatientID", how="inner")
    m3 = delta_dict["Combined"].copy()
    return m1, m2, m3

model1, model2, model3 = build_models(delta_by_roi)
print("Model 1 (Tumoral):", model1.shape)
print("Model 2 (Tumoral+Peritumoral):", model2.shape)
print("Model 3 (Combined/CPTV):", model3.shape)


## Step 6 — Fold-safe harmonizer (fixes the critical leakage issue)
Fit ONLY on training-fold data (`harmonizationLearn`); applied to the held-out test fold using
that fold's own batch/outcome values and the parameters already fixed from training
(`harmonizationApply`). No test-fold information is used to estimate the batch-effect model.


In [ ]:
class FoldSafeHarmonizer(BaseEstimator, TransformerMixin):
    """Expects X's LAST TWO columns to be [CombinedBatch, label_2yr] (passthrough, not real features).
    Imputes missing values internally (median, fit on training fold only) BEFORE calling ComBat --
    neuroHarmonize/ComBat cannot handle any NaN input (confirmed: a handful of NaN values corrupted
    its ENTIRE output in earlier testing), so this must happen inside fit()/transform(), not after."""
    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        feat_part = X_df.iloc[:, :-2].apply(pd.to_numeric)
        self.impute_medians_ = feat_part.median()
        feat_part = feat_part.fillna(self.impute_medians_).values
        covars = pd.DataFrame({
            "SITE": X_df.iloc[:, -2].values,
            "label_2yr": pd.to_numeric(X_df.iloc[:, -1]).astype(int).values
        })
        self.model_, _ = harmonizationLearn(feat_part, covars)
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X)
        feat_part = X_df.iloc[:, :-2].apply(pd.to_numeric)
        feat_part = feat_part.fillna(self.impute_medians_).values  # fold-safe: medians learned from training fold only
        covars = pd.DataFrame({
            "SITE": X_df.iloc[:, -2].values,
            "label_2yr": pd.to_numeric(X_df.iloc[:, -1]).astype(int).values
        })
        return harmonizationApply(feat_part, covars, self.model_)


## Step 7 — Pipeline: harmonize (if requested) -> scale -> filter -> RFE (matched to classifier) -> SMOTE -> classify
SMOTE now runs AFTER feature selection (fixed order). RFE's base estimator matches the classifier
being evaluated wherever the classifier exposes `feature_importances_`/`coef_`; SVM (default RBF)
is the documented exception, using Logistic for its RFE step.


In [ ]:
class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.90):
        self.threshold = threshold

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        corr = X_df.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        self.drop_idx_ = [i for i in range(len(upper.columns)) if any(upper.iloc[:, i] > self.threshold)]
        self.keep_idx_ = [i for i in range(X_df.shape[1]) if i not in self.drop_idx_]
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X)
        return X_df.iloc[:, self.keep_idx_].values


def get_classifier(name, balanced=False, pos_weight_ratio=None):
    """balanced=True applies an imbalance fallback as a SMOTE substitute for folds with too few
    minority samples to interpolate from. Each classifier uses whatever mechanism it actually
    supports: class_weight='balanced' (RF/Logistic/SVM), scale_pos_weight (XGBoost, computed from
    the fold's real class ratio via pos_weight_ratio), or sample_weight passed at fit time
    (GradientBoosting -- handled separately in get_fit_params, since sklearn's GB has no
    constructor-level weighting option)."""
    if name == "RandomForest":
        return RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, class_weight="balanced" if balanced else None)
    if name == "GradientBoosting":
        return GradientBoostingClassifier(random_state=RANDOM_STATE)
    if name == "XGBoost":
        spw = pos_weight_ratio if (balanced and pos_weight_ratio is not None) else 1
        return XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE, scale_pos_weight=spw)
    if name == "Logistic":
        return LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced" if balanced else None)
    if name == "SVM":
        return SVC(probability=True, random_state=RANDOM_STATE, class_weight="balanced" if balanced else None)
    raise ValueError(name)

def get_rfe_estimator(classifier_name):
    """RFE's estimator matches the classifier being scored, except SVM (RBF has no coef_/feature_importances_)."""
    if classifier_name == "SVM":
        return LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    return get_classifier(classifier_name)


def build_pipeline(classifier_name, harmonize, use_smote=True, y_train=None):
    """y_train is only used (to compute XGBoost's scale_pos_weight) when use_smote=False."""
    pos_weight_ratio = None
    if not use_smote and y_train is not None:
        counts = pd.Series(y_train).value_counts()
        if len(counts) == 2 and counts.min() > 0:
            pos_weight_ratio = counts.max() / counts.min()

    steps = []
    if harmonize:
        steps.append(("harmonizer", FoldSafeHarmonizer()))
    steps += [
        # Imputation now happens ONLY here, inside the pipeline, fit per-fold on training data only
        # -- the previous global pre-CV imputation (a small, disclosed leak, <0.5% of cells) has
        # been removed entirely so the "leakage-corrected" claim has no exceptions.
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("corr_filter", CorrelationFilter(threshold=CORR_THRESHOLD)),
        ("rfe", RFE(estimator=get_rfe_estimator(classifier_name), n_features_to_select=N_FEATURES_RFE, step=RFE_STEP)),
    ]
    if use_smote:
        steps.append(("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=1)))
        steps.append(("classifier", get_classifier(classifier_name, balanced=False)))
    else:
        # Fallback for folds where the minority class has <2 samples (SMOTE cannot interpolate):
        # skip SMOTE, use each classifier's own real weighting mechanism instead of a blanket
        # class_weight='balanced' that not every classifier supports.
        steps.append(("classifier", get_classifier(classifier_name, balanced=True, pos_weight_ratio=pos_weight_ratio)))
    return ImbPipeline(steps=steps)


def get_fit_params(classifier_name, use_smote, y_train):
    """GradientBoosting has no constructor-level class weighting -- it needs sample_weight passed
    at fit() time. Returns a dict suitable for **fit_params on the pipeline's .fit() call; empty
    unless this specific fallback case applies."""
    if use_smote or classifier_name != "GradientBoosting":
        return {}
    counts = pd.Series(y_train).value_counts()
    if len(counts) != 2:
        return {}
    weight_per_class = (len(y_train) / (2 * counts)).to_dict()
    sample_weight = pd.Series(y_train).map(weight_per_class).values
    return {"classifier__sample_weight": sample_weight}


## Step 8 — Metrics, SMOTE k_neighbors guard, and RepeatedStratifiedKFold evaluation

In [ ]:
def compute_metrics(y_true, y_pred, y_prob):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Sensitivity": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Specificity": recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        "Precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "Brier Score": brier_score_loss(y_true, y_prob),
    }

def safe_k_neighbors(y_train, default=5):
    """SMOTE requires k_neighbors < minority class count, and at least 2 real minority samples to
    interpolate between at all. Returns None if SMOTE should be skipped entirely for this fold
    (minority count < 2) -- caller must then fall back to class_weight='balanced' instead."""
    minority_count = pd.Series(y_train).value_counts().min()
    if minority_count < 2:
        print(f"    [guard] minority class has only {minority_count} sample(s) in this fold -- SMOTE skipped, using class_weight='balanced' instead")
        return None
    k = min(default, minority_count - 1)
    if minority_count < 6:
        print(f"    [guard] small minority class in this fold (n={minority_count}); SMOTE k_neighbors set to {k}")
    return k

def evaluate_repeated_cv(X, y, classifier_name, harmonize, collect_roc=False, track_features=False):
    rskf = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)
    fold_metrics = []
    fold_roc = []  # list of (fpr, tpr) per fold, only populated if collect_roc=True
    selected_feature_counts = {}  # feature_name -> number of folds it was selected in, if track_features=True

    for train_idx, test_idx in rskf.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        k = safe_k_neighbors(y_train)
        pipeline = build_pipeline(classifier_name, harmonize, use_smote=(k is not None), y_train=y_train)
        if k is not None:
            pipeline.set_params(smote__k_neighbors=k)

        fit_params = get_fit_params(classifier_name, use_smote=(k is not None), y_train=y_train)
        try:
            pipeline.fit(X_train, y_train, **fit_params)
            y_pred = pipeline.predict(X_test)
            y_prob = pipeline.predict_proba(X_test)[:, 1]
        except ValueError as e:
            if harmonize and "not aligned" in str(e):
                print(f"    [guard] harmonization design-matrix mismatch this fold -- falling back to un-harmonized features for this fold only")
                X_train_fb, X_test_fb = X_train.iloc[:, :-2], X_test.iloc[:, :-2]
                k2 = safe_k_neighbors(y_train)
                fb_pipeline = build_pipeline(classifier_name, harmonize=False, use_smote=(k2 is not None), y_train=y_train)
                if k2 is not None:
                    fb_pipeline.set_params(smote__k_neighbors=k2)
                fb_fit_params = get_fit_params(classifier_name, use_smote=(k2 is not None), y_train=y_train)
                fb_pipeline.fit(X_train_fb, y_train, **fb_fit_params)
                y_pred = fb_pipeline.predict(X_test_fb)
                y_prob = fb_pipeline.predict_proba(X_test_fb)[:, 1]
            else:
                raise
        fold_metrics.append(compute_metrics(y_test, y_pred, y_prob))

        if collect_roc:
            from sklearn.metrics import roc_curve
            fpr, tpr, _ = roc_curve(y_test, y_prob)
            fold_roc.append((fpr, tpr))

        if track_features:
            feature_names = list(X_train.columns[:-2]) if harmonize else list(X_train.columns)
            keep_idx = pipeline.named_steps["corr_filter"].keep_idx_
            names_after_filter = [feature_names[i] for i in keep_idx]
            rfe_support = pipeline.named_steps["rfe"].support_
            selected = [names_after_filter[i] for i in range(len(names_after_filter)) if rfe_support[i]]
            for feat in selected:
                selected_feature_counts[feat] = selected_feature_counts.get(feat, 0) + 1

    mdf = pd.DataFrame(fold_metrics)
    means, stds = mdf.mean(), mdf.std()
    formatted = {k: f"{means[k]:.3f} +/- {stds[k]:.3f}" for k in means.index}
    raw_means = means.to_dict()
    raw_stds = stds.to_dict()
    return formatted, raw_means, raw_stds, fold_roc, selected_feature_counts


def get_training_metrics(X, y, classifier_name, harmonize):
    """Fits ONE pipeline on all data, scores on that same data -- in-sample fit, not a
    generalization estimate. Reported alongside CV metrics to show the overfitting gap."""
    k = safe_k_neighbors(y)
    pipeline = build_pipeline(classifier_name, harmonize, use_smote=(k is not None), y_train=y)
    if k is not None:
        pipeline.set_params(smote__k_neighbors=k)
    fit_params = get_fit_params(classifier_name, use_smote=(k is not None), y_train=y)
    pipeline.fit(X, y, **fit_params)
    y_pred = pipeline.predict(X)
    y_prob = pipeline.predict_proba(X)[:, 1]
    return compute_metrics(y, y_pred, y_prob)


## Step 9 — Run all datasets x classifiers: BOTH training and CV scores, for every metric
Produces one detailed CSV with Training and CV (mean +/- std across 50 folds) side by side for
every metric, for all 30 dataset/classifier/harmonization combinations. Also collects per-fold ROC
points for the RandomForest + Harmonized runs (your best-performing classifier) across Model 1/2/3,
used to build a proper CV-based mean ROC curve with a variability band in Step 10.


In [ ]:
classifiers = ["RandomForest", "GradientBoosting", "XGBoost", "Logistic", "SVM"]
model_frames = {"Model1_Tumoral": model1, "Model2_TumoralPlusPeritumoral": model2, "Model3_Combined_CPTV": model3}

detailed_rows = []
auc_lookup = {}
roc_data = {}       # only populated for RandomForest + Harmonized, per model -- used for the CV ROC plot
feature_stability = {}  # same subset -- tracks how often each feature was selected across the 50 folds

for mname, mframe in model_frames.items():
    y = mframe["label_2yr"].astype(int)
    X_plain = mframe.drop(columns=["INFO_PatientID", "label_2yr", "CombinedBatch"])
    X_with_batch = pd.concat([X_plain, mframe[["CombinedBatch", "label_2yr"]].reset_index(drop=True)], axis=1)

    for clf_name in classifiers:
        for harmonize, X_eval, tag in [(False, X_plain, "NonHarmonized"), (True, X_with_batch, "Harmonized")]:
            label = f"{mname}_{tag}"
            print(f"Running {mname} {tag} + {clf_name} ...")

            cv_formatted, cv_means, cv_stds, _, _ = evaluate_repeated_cv(X_eval, y, clf_name, harmonize)
            train_metrics = get_training_metrics(X_eval, y, clf_name, harmonize)

            for metric in train_metrics:
                detailed_rows.append({
                    "Dataset": label, "Classifier": clf_name, "Metric": metric,
                    "Training": round(train_metrics[metric], 3),
                    "CV_Mean": round(cv_means[metric], 3),
                    "CV_Std": round(cv_stds[metric], 3),
                    "CV_Formatted": cv_formatted[metric],
                    "Train_CV_Gap": round(train_metrics[metric] - cv_means[metric], 3),
                })

            auc_lookup[(label, clf_name)] = cv_means["ROC-AUC"]

detailed_df = pd.DataFrame(detailed_rows)
detailed_df.to_csv("detailed_train_cv_metrics.csv", index=False)
print("\nSaved detailed_train_cv_metrics.csv")
detailed_df.head(20)


## Step 9b — Identify the ACTUAL best classifier+harmonization per model, then re-run ONLY those
The earlier version of this notebook hardcoded RandomForest+Harmonized for the feature-stability
table and ROC curve, asserting it was "the best classifier" without checking `auc_lookup`. This
step determines the real winner per model type from the data, then re-runs `evaluate_repeated_cv`
once more for each of those 3 verified winners (with feature-tracking and ROC-collection turned
on) -- a small, deliberate extra cost, but necessary so the plots below are guaranteed to describe
the actual best-performing configuration, not an assumed one.


In [ ]:
def get_best_config_per_model(auc_source, source_type="auc_lookup"):
    """Single source of truth for 'best classifier+harmonization per model type', used by both
    Step 9b (feature stability / ROC) and Step 11 (confusion matrices) -- consolidated so the two
    can't silently disagree if one is changed later without the other."""
    best = {}
    if source_type == "auc_lookup":
        for mname in model_frames:
            candidates = {(label, clf): auc for (label, clf), auc in auc_source.items()
                          if label.replace("_Harmonized", "").replace("_NonHarmonized", "") == mname}
            best_label_clf = max(candidates, key=candidates.get)
            best[mname] = (best_label_clf[0], best_label_clf[1], candidates[best_label_clf])
    else:  # detailed_df
        auc_rows = auc_source[auc_source["Metric"] == "ROC-AUC"].copy()
        auc_rows["ModelType"] = auc_rows["Dataset"].str.replace("_Harmonized", "").str.replace("_NonHarmonized", "")
        top = auc_rows.loc[auc_rows.groupby("ModelType")["CV_Mean"].idxmax()]
        for _, row in top.iterrows():
            best[row["ModelType"]] = (row["Dataset"], row["Classifier"], row["CV_Mean"])
    return best


best_config_per_model_raw = get_best_config_per_model(auc_lookup, source_type="auc_lookup")
best_config_per_model = {}
for mname, (best_label, best_clf, best_auc) in best_config_per_model_raw.items():
    best_config_per_model[mname] = (best_label, best_clf)
    print(f"{mname}: verified best = {best_clf} on {best_label} (AUC={best_auc:.3f})")

roc_data = {}
feature_stability = {}
for mname, (best_label, best_clf) in best_config_per_model.items():
    print(f"\nRe-running {best_label} + {best_clf} once more to collect ROC/feature-stability data ...")
    harmonize = best_label.endswith("_Harmonized")
    mframe = model_frames[mname]
    y = mframe["label_2yr"].astype(int)
    X_plain = mframe.drop(columns=["INFO_PatientID", "label_2yr", "CombinedBatch"])
    X_eval = pd.concat([X_plain, mframe[["CombinedBatch", "label_2yr"]].reset_index(drop=True)], axis=1) if harmonize else X_plain

    _, _, _, fold_roc, feat_counts = evaluate_repeated_cv(X_eval, y, best_clf, harmonize, collect_roc=True, track_features=True)
    roc_data[mname] = (fold_roc, best_label, best_clf)
    feature_stability[mname] = (feat_counts, best_label, best_clf)


## Step 9c — RFE feature-selection stability across folds (for each model's VERIFIED best config)
Shows how often each feature was among the top-3 selected across the 50 CV folds. A "top 3
features" claim is only meaningful if the same features keep getting picked; if selection is
scattered across many different features fold-to-fold, that is itself evidence the signal is
closer to noise-fitting than a real, stable biomarker.


In [ ]:
for mname, (counts, best_label, best_clf) in feature_stability.items():
    total_folds = N_SPLITS * N_REPEATS
    harmonize_tag = "Harmonized" if best_label.endswith("Harmonized") else "NonHarmonized"
    stability_df = pd.DataFrame(
        [(feat, n, round(n/total_folds, 3)) for feat, n in sorted(counts.items(), key=lambda x: -x[1])],
        columns=["Feature", "Times_Selected", "Selection_Frequency"]
    )
    print(f"\n=== Feature stability, {mname} ({best_clf}, {harmonize_tag}), out of {total_folds} folds ===")
    print(stability_df.head(10).to_string(index=False))
    stability_df.to_csv(f"feature_stability_{mname}.csv", index=False)


## Step 10 — CV-based mean ROC curve with variability band
Built correctly from the actual repeated-CV folds (not a separate single-pass run): each of the 50
folds' own ROC curve is interpolated onto a common set of FPR points, then averaged -- giving one
mean curve whose AUC matches your reported repeated-CV mean, with a shaded band showing the real
fold-to-fold variability. Shown for each model's VERIFIED best classifier+harmonization (from Step
9b), not an assumed one.


In [ ]:
mean_fpr = np.linspace(0, 1, 100)

plt.figure(figsize=(8, 8))
for mname, (fold_roc, best_label, best_clf) in roc_data.items():
    interp_tprs = []
    for fpr, tpr in fold_roc:
        interp_tpr = np.interp(mean_fpr, fpr, tpr)
        interp_tpr[0] = 0.0
        interp_tprs.append(interp_tpr)

    interp_tprs = np.array(interp_tprs)
    mean_tpr = interp_tprs.mean(axis=0)
    mean_tpr[-1] = 1.0
    std_tpr = interp_tprs.std(axis=0)

    mean_auc = auc_lookup[(best_label, best_clf)]
    harmonize_tag = best_label.split("_")[-1]

    plt.plot(mean_fpr, mean_tpr, label=f"{mname} [{best_clf}, {harmonize_tag}] (AUC={mean_auc:.3f})")
    plt.fill_between(mean_fpr, np.maximum(mean_tpr - std_tpr, 0), np.minimum(mean_tpr + std_tpr, 1), alpha=0.15)

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance")
plt.xlabel("1 - Specificity (False Positive Rate)")
plt.ylabel("Sensitivity (True Positive Rate)")
plt.title("Mean ROC Curve +/- SD across repeated CV folds")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("cv_roc_curves.png", dpi=150)
plt.show()
print("Saved cv_roc_curves.png")


## Step 11 — Confusion matrices for the top-performing configuration per ROI type
Identifies the best classifier+harmonization combination for each of Model 1/2/3 (by CV mean
ROC-AUC from Step 9's table), then builds a confusion matrix using out-of-fold predictions from a
clean 5-fold pass -- genuinely held-out, not training predictions -- at the default 0.5 threshold.


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def get_oof_predictions(X, y, classifier_name, harmonize):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    oof_pred = np.zeros(len(y), dtype=int)
    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train = y.iloc[train_idx]
        k = safe_k_neighbors(y_train)
        pipeline = build_pipeline(classifier_name, harmonize, use_smote=(k is not None), y_train=y_train)
        if k is not None:
            pipeline.set_params(smote__k_neighbors=k)
        fit_params = get_fit_params(classifier_name, use_smote=(k is not None), y_train=y_train)
        pipeline.fit(X_train, y_train, **fit_params)
        oof_pred[test_idx] = pipeline.predict(X_test)
    return oof_pred

# Uses the SAME best_config_per_model already computed in Step 9b (via the shared
# get_best_config_per_model function), rather than re-deriving it independently -- avoids the
# two steps silently disagreeing if one is changed later without the other.
print("Top configuration per model type (from Step 9b):")
for mname, (best_label, best_clf) in best_config_per_model.items():
    print(f"  {mname}: {best_clf} on {best_label}")

fig, axes = plt.subplots(1, len(best_config_per_model), figsize=(6*len(best_config_per_model), 5))
if len(best_config_per_model) == 1:
    axes = [axes]

for ax, (mname, (best_label, clf_name)) in zip(axes, best_config_per_model.items()):
    harmonize = best_label.endswith("_Harmonized")
    harmonize_label = "Harmonized" if harmonize else "NonHarmonized"
    mframe = model_frames[mname]
    y = mframe["label_2yr"].astype(int)
    X_plain = mframe.drop(columns=["INFO_PatientID", "label_2yr", "CombinedBatch"])
    X_eval = pd.concat([X_plain, mframe[["CombinedBatch", "label_2yr"]].reset_index(drop=True)], axis=1) if harmonize else X_plain

    oof_pred = get_oof_predictions(X_eval, y, clf_name, harmonize)
    cm = confusion_matrix(y, oof_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Not survived", "Survived"])
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(f"{mname}\n{clf_name} ({harmonize_label})")

plt.tight_layout()
plt.savefig("confusion_matrices_top_models.png", dpi=150)
plt.show()
print("Saved confusion_matrices_top_models.png")


## Step 12 — Permutation test on the top result, with multiple-comparisons correction
Two real problems in the original version of this test, per external review, both addressed here:

1. **A full "max-statistic" permutation test** (shuffle labels once, rerun ALL 30 comparisons,
   record the best AUC, repeat many times) is the technically correct way to test "is the best of
   30 comparisons better than chance." It is also computationally infeasible here: 500+ permutations
   x 30 combinations x 50 folds would mean tens of millions of fold-fits -- not realistic on a
   laptop before the deadline. Instead, this runs a properly-powered single-arm permutation test
   (`N_PERMUTATIONS = 500`, giving p-value resolution to ~0.002) on the winning configuration, then
   applies a **Bonferroni correction** (multiplying the raw p-value by 30, the number of
   comparisons actually run in Step 9) as a conservative approximation. This is disclosed here as an
   approximation, not the theoretically ideal test, because that ideal test is not feasible to run.
2. **Resolution floor**: with N permutations, the smallest possible p-value is `1/(N+1)`. At
   `N_PERMUTATIONS = 500` that floor is ~0.002 -- well below both the raw 0.05 threshold and the
   Bonferroni-corrected 0.05/30 ≈ 0.0017 threshold, unlike the earlier N=20 version (floor ≈0.048,
   which could never actually demonstrate significance at any reasonable threshold).

**Runtime note**: 500 permutations x 50 folds for one configuration is substantial but far more
tractable than the full max-statistic version -- expect this to take real time, but not the
many-hours cost the original N=100 estimate implied once RFE's step=10 fix is accounted for.


In [ ]:
best_key = max(auc_lookup, key=auc_lookup.get)
best_dataset, best_classifier = best_key
n_comparisons = len(auc_lookup)  # should be 30 (3 models x 2 harmonization states x 5 classifiers)
print(f"Best configuration: {best_dataset} + {best_classifier}, real mean AUC = {auc_lookup[best_key]:.3f}")
print(f"Number of comparisons run in Step 9 (for Bonferroni correction): {n_comparisons}")

harmonize_best = best_dataset.endswith("_Harmonized")
base_mname = best_dataset.replace("_Harmonized", "").replace("_NonHarmonized", "")
mframe = model_frames[base_mname]
y_real = mframe["label_2yr"].astype(int)
X_plain = mframe.drop(columns=["INFO_PatientID", "label_2yr", "CombinedBatch"])
X_eval = pd.concat([X_plain, mframe[["CombinedBatch", "label_2yr"]].reset_index(drop=True)], axis=1) if harmonize_best else X_plain

N_PERMUTATIONS = 500   # gives p-value resolution to ~0.002; reduce if this proves too slow, but
                       # note N=20 (the earlier version) had a resolution floor of ~0.048 and could
                       # never demonstrate significance at any real threshold -- this must stay >=100.
rng = np.random.RandomState(RANDOM_STATE)
null_aucs = []
for i in range(N_PERMUTATIONS):
    y_shuffled = pd.Series(rng.permutation(y_real.values), index=y_real.index)
    _, cv_means, _, _, _ = evaluate_repeated_cv(X_eval, y_shuffled, best_classifier, harmonize=harmonize_best)
    null_aucs.append(cv_means["ROC-AUC"])
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{N_PERMUTATIONS} permutations done")

null_aucs = np.array(null_aucs)
real_auc = auc_lookup[best_key]
raw_p_value = (np.sum(null_aucs >= real_auc) + 1) / (N_PERMUTATIONS + 1)
bonferroni_p_value = min(raw_p_value * n_comparisons, 1.0)

print(f"\nReal AUC: {real_auc:.3f}")
print(f"Null distribution: mean={null_aucs.mean():.3f}, std={null_aucs.std():.3f}")
print(f"Raw permutation p-value: {raw_p_value:.4f}")
print(f"Bonferroni-corrected p-value (x{n_comparisons} comparisons): {bonferroni_p_value:.4f}")
print(f"\nNote: this is an approximation (single-arm test + Bonferroni), not a full max-statistic")
print(f"permutation over all {n_comparisons} comparisons, which was computationally infeasible here.")
print(f"Note: Bonferroni assumes independent comparisons; these {n_comparisons} comparisons share")
print(f"overlapping patients and feature sets across Model 1/2/3, so the true correction is likely")
print(f"less conservative than Bonferroni -- if the corrected p-value clears significance, trust it;")
print(f"if it narrowly misses, the real answer may still be significant, not necessarily a null result.")
